# ModelShield — Phase 1: An AI Firewall for Adversarial Attacks
**Course Subject:** Intelligent Model Design (IMD)  
**Project Role:** Senior Machine Learning Engineer & AI Security Architect  

---

### Executive Overview
**ModelShield** is an AI cybersecurity layer designed to detect and intercept adversarially modified images before they reach downstream deep learning classifiers. Adversarial examples contain imperceptible, mathematically calculated perturbations that intentionally induce high-confidence classification errors.

In **Phase 1**, this notebook demonstrates:
1. **Module 1 (Classifier):** Training a custom CNN to classify 10-class CIFAR-10 images.
2. **Module 2 (Attack Engine):** Generating Fast Gradient Sign Method (FGSM) adversarial perturbations across multiple epsilons (0.01, 0.03, 0.05, 0.10) and evaluating Attack Success Rate (ASR).
3. **Module 3 (Firewall Detector):** Generating balanced clean-vs-adversarial datasets without data leakage and training a dedicated binary CNN detector.
4. **Module 4 (Integrated AI Firewall):** Implementing real-time security inspection, adversarial probability gating, and structured decision reporting (`ALLOWED` vs `BLOCKED`).

---

### Google Colab Quick-Start Instructions
1. **Enable GPU Hardware Acceleration:** Navigate to `Runtime` -> `Change runtime type` -> Set `Hardware accelerator` to **GPU (T4)**.
2. **Clone the Repository:**
```bash
!git clone https://github.com/<your-username>/ModelShield.git
%cd ModelShield
```
3. Run all cells sequentially from top to bottom (`Runtime` -> `Run all`).


## 1. Environment & Version Verification


In [ ]:
import sys
import torch
import torchvision

print(f"Python Runtime:     {sys.version.split()[0]}")
print(f"PyTorch Version:    {torch.__version__}")
print(f"Torchvision:        {torchvision.__version__}")


## 2. Hardware Acceleration & CUDA Diagnostics


In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Active Device: {device}")

if torch.cuda.is_available():
    print(f"GPU Device Name:    {torch.cuda.get_device_name(0)}")
    print(f"Total GPU Memory:   {torch.cuda.get_device_properties(0).total_memory / (1024**3):.2f} GB")
    print(f"Allocated Memory:   {torch.cuda.memory_allocated(0) / (1024**2):.2f} MB")
else:
    print("Notice: CUDA is unavailable. Execution will fall back to CPU.")


## 3. Dependency Installation


In [ ]:
from pathlib import Path

# Install project dependencies
req_file = Path("requirements.txt")
if req_file.exists():
    !pip install -r requirements.txt --quiet
else:
    !pip install torch torchvision numpy matplotlib scikit-learn pillow tqdm pytest --quiet

print("Dependencies successfully installed and verified!")


## 4. Google Drive Persistence (Optional)
Toggle `MOUNT_DRIVE = True` to synchronize model checkpoints and experiment artifacts directly to Google Drive.


In [ ]:
import os
from pathlib import Path

MOUNT_DRIVE = False  # Set to True in Google Colab to preserve trained weights
DRIVE_CHECKPOINT_DIR = None

if MOUNT_DRIVE:
    try:
        from google.colab import drive
        drive.mount('/content/drive')
        DRIVE_CHECKPOINT_DIR = Path('/content/drive/MyDrive/ModelShield_Checkpoints')
        DRIVE_CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)
        print(f"Google Drive mounted. Checkpoints will be mirrored to: {DRIVE_CHECKPOINT_DIR}")
    except Exception as e:
        print(f"Google Drive mount skipped: {e}")
else:
    print("Local persistence enabled: Checkpoints will be saved in ./checkpoints/.")


## 5. Import Modular Codebase & Deterministic Seed Setting


In [ ]:
import sys
from pathlib import Path

# Ensure root directory is on Python search path
PROJECT_ROOT = Path.cwd()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import config
config.set_seed(config.RANDOM_SEED)

from src.data import (
    get_cifar10_dataloaders,
    denormalize_tensor,
    normalize_tensor,
    AdversarialDataset
)
from src.models import CIFAR10Classifier, AdversarialDetector
from src.attacks import (
    fgsm_attack,
    evaluate_attack_success_rate,
    visualize_adversarial_examples
)
from src.train import (
    train_classifier,
    evaluate_classifier_test_set,
    build_detector_dataset,
    train_detector,
    evaluate_detector_test_set,
    plot_training_history
)
from src.firewall import ModelShieldFirewall

print("ModelShield modules imported successfully with deterministic seed:", config.RANDOM_SEED)


## 6. CIFAR-10 Ingestion, Preprocessing & Non-Leaking Splits
Downloads the CIFAR-10 dataset.
- Training Split: 40,000 images (augmented with random crop & horizontal flip)
- Validation Split: 10,000 images (eval normalization)
- Untouched Test Split: 10,000 images (held out until final model evaluation)


In [ ]:
batch_size = 128 if torch.cuda.is_available() else config.CLASSIFIER_BATCH_SIZE

train_loader, val_loader, test_loader = get_cifar10_dataloaders(
    data_dir=config.DATA_DIR,
    batch_size=batch_size,
    val_split_ratio=config.VAL_SPLIT_RATIO,
    random_seed=config.RANDOM_SEED,
    num_workers=2 if torch.cuda.is_available() else 0,
    download=True
)

print(f"Train Dataset:       {len(train_loader.dataset):,} images ({len(train_loader)} batches)")
print(f"Validation Dataset:  {len(val_loader.dataset):,} images ({len(val_loader)} batches)")
print(f"Untouched Test Set:  {len(test_loader.dataset):,} images ({len(test_loader)} batches)")
print(f"Classes ({config.NUM_CLASSES}): {config.CIFAR10_CLASSES}")


## 7. Module 1: Train Custom CNN Classifier
Trains our custom multi-layer CNN classifier.
Uses raw logits with `nn.CrossEntropyLoss` and the Adam optimizer. Checkpoints the best model based on validation accuracy.


In [ ]:
classifier = CIFAR10Classifier(num_classes=config.NUM_CLASSES).to(device)

epochs = 15 if torch.cuda.is_available() else 3

history_classifier = train_classifier(
    model=classifier,
    train_loader=train_loader,
    val_loader=val_loader,
    device=device,
    epochs=epochs,
    lr=config.CLASSIFIER_LR,
    weight_decay=config.CLASSIFIER_WEIGHT_DECAY,
    checkpoint_path=config.CLASSIFIER_CHECKPOINT_PATH
)

# Plot loss and accuracy curves
fig_cls = plot_training_history(
    history=history_classifier,
    title="CIFAR-10 Classifier",
    save_path=config.OUTPUTS_DIR / "classifier_training_history.png"
)


## 8. Evaluate Classifier on Untouched Test Set
Evaluates the best classifier checkpoint on the untouched 10,000-image test set.
Generates an executive classification report and confusion matrix.


In [ ]:
# Load best checkpoint
ckpt = torch.load(config.CLASSIFIER_CHECKPOINT_PATH, map_location=device, weights_only=True)
classifier.load_state_dict(ckpt["model_state_dict"])
classifier.eval()

cls_results = evaluate_classifier_test_set(
    model=classifier,
    test_loader=test_loader,
    device=device,
    save_cm_path=config.OUTPUTS_DIR / "classifier_confusion_matrix.png"
)

print(f"\nUntouched Test Set Overall Accuracy: {cls_results['test_accuracy'] * 100:.2f}%\n")
print("Classification Report:")
print(cls_results["classification_report"])


## 9. Module 2: FGSM Adversarial Attack Demonstration
Applies the Fast Gradient Sign Method (FGSM):
$$x_{adv} = \text{clip}(x + \epsilon \cdot \text{sign}(\nabla_x \mathcal{L}(f(x), y)), 0, 1)$$

Perturbations are performed in the unnormalized $[0, 1]$ image space with exact channel-specific standard deviation handling.


In [ ]:
sample_images, sample_labels = next(iter(test_loader))
test_image = sample_images[0]
test_label = sample_labels[0].item()

print(f"Selected Clean Test Sample:")
print(f"  - Ground Truth Class: {config.CIFAR10_CLASSES[test_label]} (Class ID: {test_label})")

for eps in config.FGSM_EPSILONS:
    adv_tensor = fgsm_attack(
        model=classifier,
        images=test_image.unsqueeze(0).to(device),
        labels=torch.tensor([test_label], device=device),
        epsilon=eps
    )
    with torch.no_grad():
        logits = classifier(adv_tensor)
        pred = logits.argmax(dim=1).item()
    fooled = (pred != test_label)
    status_str = "FOOLED" if fooled else "CORRECT"
    print(f"  - Epsilon {eps:.2f} -> Prediction: {config.CIFAR10_CLASSES[pred]:<12} [{status_str}]")


## 10. Visual Comparison: Clean vs Adversarial Examples


In [ ]:
fig_comparison = visualize_adversarial_examples(
    model=classifier,
    sample_image=test_image,
    sample_label=test_label,
    epsilons=config.FGSM_EPSILONS,
    device=device,
    save_path=str(config.OUTPUTS_DIR / "fgsm_visual_comparison.png")
)
import matplotlib.pyplot as plt
plt.show()


## 11. Calculate Attack Success Rate (ASR) Across Epsilon Spectrum
Computes untargeted Attack Success Rate where the denominator consists strictly of images that were originally correctly classified:
$$\text{ASR} = \frac{\text{Number of Samples Fooled}}{\text{Samples Originally Correct}}$$


In [ ]:
import matplotlib.pyplot as plt

asr_eval_results = []
eps_range = [0.0, 0.01, 0.03, 0.05, 0.10]

print(f"{'Epsilon':<8} | {'Clean Acc':<12} | {'Adv Acc':<12} | {'ASR (%)':<10}")
print("-" * 50)

for eps in eps_range:
    eval_res = evaluate_attack_success_rate(
        model=classifier,
        dataloader=test_loader,
        epsilon=eps,
        device=device,
        max_samples=5000 if torch.cuda.is_available() else 1000
    )
    asr_eval_results.append(eval_res)
    print(f"{eval_res['epsilon']:<8.2f} | {eval_res['clean_accuracy']*100:<11.2f}% | "
          f"{eval_res['adversarial_accuracy']*100:<11.2f}% | {eval_res['attack_success_rate']*100:<9.2f}%")

# Plot ASR curve
plt.figure(figsize=(7, 4))
eps_points = [r["epsilon"] for r in asr_eval_results]
asr_points = [r["attack_success_rate"] * 100 for r in asr_eval_results]
plt.plot(eps_points, asr_points, "r-o", linewidth=2.5, markersize=6)
plt.title("Attack Success Rate (ASR) vs Epsilon Magnitude")
plt.xlabel("Epsilon (Perturbation Magnitude)")
plt.ylabel("Attack Success Rate (%)")
plt.grid(True, alpha=0.3)
plt.savefig(config.OUTPUTS_DIR / "asr_curve.png", dpi=300, bbox_inches="tight")
plt.show()


## 12. Module 3: Generate Adversarial Detection Dataset
Builds balanced binary datasets:
- **Class 0:** Clean CIFAR-10 images
- **Class 1:** Adversarially perturbed FGSM images (epsilon = 0.03)

Strictly generated per split (train, val, test) to eliminate data leakage.


In [ ]:
detector_eps = config.DEFAULT_EPSILON
print(f"Generating balanced detector splits at epsilon = {detector_eps}...")

# Use full datasets on GPU, sample limits on CPU
max_tr = None if torch.cuda.is_available() else 2000
max_v  = None if torch.cuda.is_available() else 500
max_ts = None if torch.cuda.is_available() else 1000

detector_train_loader = build_detector_dataset(
    classifier=classifier,
    clean_loader=train_loader,
    epsilon=detector_eps,
    device=device,
    max_clean_samples=max_tr
)

detector_val_loader = build_detector_dataset(
    classifier=classifier,
    clean_loader=val_loader,
    epsilon=detector_eps,
    device=device,
    max_clean_samples=max_v
)

detector_test_loader = build_detector_dataset(
    classifier=classifier,
    clean_loader=test_loader,
    epsilon=detector_eps,
    device=device,
    max_clean_samples=max_ts
)

print(f"Detector Train Samples: {len(detector_train_loader.dataset):,} (50% Clean, 50% Adv)")
print(f"Detector Val Samples:   {len(detector_val_loader.dataset):,} (50% Clean, 50% Adv)")
print(f"Detector Test Samples:  {len(detector_test_loader.dataset):,} (50% Clean, 50% Adv)")


## 13. Train Adversarial Detection CNN
Trains a custom binary classification CNN using `BCEWithLogitsLoss`.
The model learns to detect structural perturbation artifacts.


In [ ]:
detector = AdversarialDetector().to(device)

detector_epochs = 10 if torch.cuda.is_available() else 3

history_detector = train_detector(
    detector=detector,
    train_loader=detector_train_loader,
    val_loader=detector_val_loader,
    device=device,
    epochs=detector_epochs,
    lr=config.DETECTOR_LR,
    weight_decay=config.DETECTOR_WEIGHT_DECAY,
    checkpoint_path=config.DETECTOR_CHECKPOINT_PATH,
    threshold=config.DETECTOR_THRESHOLD
)

fig_det = plot_training_history(
    history=history_detector,
    title="Adversarial Detector",
    save_path=config.OUTPUTS_DIR / "detector_training_history.png"
)


## 14. Evaluate Detector on Held-out Adversarial Test Split
Calculates Accuracy, Precision, Recall, F1-Score, False Positive Rate (FPR), and Confusion Matrix.


In [ ]:
# Load best detector checkpoint
det_ckpt = torch.load(config.DETECTOR_CHECKPOINT_PATH, map_location=device, weights_only=True)
detector.load_state_dict(det_ckpt["model_state_dict"])
detector.eval()

detector_eval = evaluate_detector_test_set(
    detector=detector,
    test_loader=detector_test_loader,
    device=device,
    threshold=config.DETECTOR_THRESHOLD,
    save_cm_path=config.OUTPUTS_DIR / "detector_confusion_matrix.png"
)

print("\n==================================================")
print("     ADVERSARIAL DETECTOR EVALUATION REPORT       ")
print("==================================================")
print(f"Accuracy:            {detector_eval['accuracy'] * 100:.2f}%")
print(f"Precision:           {detector_eval['precision'] * 100:.2f}%")
print(f"Recall:              {detector_eval['recall'] * 100:.2f}%")
print(f"F1-Score:            {detector_eval['f1_score']:.4f}")
print(f"False Positive Rate: {detector_eval['false_positive_rate'] * 100:.2f}%")
print(f"True Negatives (TN): {detector_eval['true_negatives']}")
print(f"False Positives(FP): {detector_eval['false_positives']}")
print(f"False Negatives(FN): {detector_eval['false_negatives']}")
print(f"True Positives (TP): {detector_eval['true_positives']}")
print("==================================================")


## 15. Module 4: Integrated ModelShield Firewall Pipeline
Instantiates the end-to-end security firewall.
Inspects incoming images, computes adversarial probability, and enforces decision gating:
- $P(\text{adv}) < \text{threshold} \implies \textbf{ALLOWED} \to$ Forwarded to classifier
- $P(\text{adv}) \ge \text{threshold} \implies \textbf{BLOCKED} \to$ Quarantined as suspicious


In [ ]:
firewall = ModelShieldFirewall(
    classifier=classifier,
    detector=detector,
    threshold=config.DETECTOR_THRESHOLD,
    device=device
)

# Test 1: Clean sample verification
print("--- [Firewall Test 1: Clean Image Evaluation] ---")
clean_img_tensor = sample_images[2].to(device)
clean_result = firewall.process(clean_img_tensor)
print(firewall.format_security_report(clean_result, input_name="clean_sample_eval.png"))

# Test 2: FGSM adversarial attack verification
print("\n--- [Firewall Test 2: Adversarial Image Inspection] ---")
adv_img_tensor = fgsm_attack(
    model=classifier,
    images=clean_img_tensor.unsqueeze(0),
    labels=sample_labels[2].unsqueeze(0).to(device),
    epsilon=0.03
)
adv_result = firewall.process(adv_img_tensor)
print(firewall.format_security_report(adv_result, input_name="adversarial_sample_eval.png"))


## 16. Checkpoint Persistence and Reload Verification


In [ ]:
import shutil

print(f"Classifier Checkpoint: {config.CLASSIFIER_CHECKPOINT_PATH} (Exists: {config.CLASSIFIER_CHECKPOINT_PATH.exists()})")
print(f"Detector Checkpoint:   {config.DETECTOR_CHECKPOINT_PATH} (Exists: {config.DETECTOR_CHECKPOINT_PATH.exists()})")

if MOUNT_DRIVE and DRIVE_CHECKPOINT_DIR:
    shutil.copy(config.CLASSIFIER_CHECKPOINT_PATH, DRIVE_CHECKPOINT_DIR / config.CLASSIFIER_CHECKPOINT_NAME)
    shutil.copy(config.DETECTOR_CHECKPOINT_PATH, DRIVE_CHECKPOINT_DIR / config.DETECTOR_CHECKPOINT_NAME)
    print(f"Checkpoints backed up to Google Drive at: {DRIVE_CHECKPOINT_DIR}")

# Test reload from checkpoint
reloaded_firewall = ModelShieldFirewall(
    classifier_path=config.CLASSIFIER_CHECKPOINT_PATH,
    detector_path=config.DETECTOR_CHECKPOINT_PATH,
    threshold=config.DETECTOR_THRESHOLD,
    device=device
)
assert reloaded_firewall.classifier_loaded, "Classifier weights failed to load"
assert reloaded_firewall.detector_loaded, "Detector weights failed to load"
print("Model reload verification: PASSED!")


## 17. Experiment Summary & Generated Output Artifacts


In [ ]:
artifacts = sorted(list(config.OUTPUTS_DIR.glob('*')))
print(f"Generated {len(artifacts)} artifact files in {config.OUTPUTS_DIR}:")
for f in artifacts:
    print(f"  - {f.name}")

summary_content = f"""==================================================
MODELSHIELD PHASE 1 EXPERIMENTAL SUMMARY
==================================================
Compute Device:              {device}
Classifier Untouched Test Accuracy: {cls_results['test_accuracy'] * 100:.2f}%
Detector Test Accuracy:      {detector_eval['accuracy'] * 100:.2f}%
Detector Precision:          {detector_eval['precision'] * 100:.2f}%
Detector Recall:             {detector_eval['recall'] * 100:.2f}%
Detector F1-Score:           {detector_eval['f1_score']:.4f}
Detector False Positive Rate:{detector_eval['false_positive_rate'] * 100:.2f}%
Firewall Decision Threshold: {config.DETECTOR_THRESHOLD}
==================================================
"""
summary_file = config.OUTPUTS_DIR / "experiment_summary.txt"
summary_file.write_text(summary_content)
print(f"\n{summary_content}")
print(f"Executive summary persisted to: {summary_file}")
